In [1]:
import torch
import numpy as np
from typing import List, Optional, Dict
import matplotlib.pyplot as plt
import seaborn as sns
from torchtext.data.metrics import bleu_score
from train import (
    DecoderTransformer, tiny_stories_vocab, embedding_layer,
    create_decoder_transformer
)
import spacy
from tqdm import tqdm

from datasets import load_dataset, concatenate_datasets, load_from_disk, DownloadMode

/home/anwesh/scratch/miniconda3/envs/pt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/anwesh/scratch/miniconda3/envs/pt/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/anwesh/scratch/miniconda3/envs/pt/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWa

Tokenized dataset loaded from disk.


In [2]:
class TextGenerator:
    def __init__(self, model_path: str, device: str = 'cuda:1'):
        # Load trained model
        checkpoint = torch.load(model_path, map_location=device)
        self.model = create_decoder_transformer(
            seq_len=checkpoint['config']['seq_len'],
            device=device,
            pretrained_embeddings=embedding_layer
        )
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        self.device = device
        self.max_seq_len = checkpoint['config']['seq_len']  # Store max sequence length

    def forward_with_attention(self, input_ids):
        """Forward pass that captures attention weights"""
        # Get embeddings
        x = self.model.embedding(input_ids)
        if self.model.embedding_projection is not None:
            x = self.model.embedding_projection(x)
        x = self.model.positional_encoding(x)
        
        attention_weights = []
        # Process through layers
        for layer in self.model.layers:
            # Process through self attention
            q = layer.self_attn.linear_q(x)
            k = layer.self_attn.linear_k(x)
            v = layer.self_attn.linear_v(x)
            
            # Reshape for multi-head attention
            batch_size = q.size(0)
            q = q.view(batch_size, -1, layer.self_attn.num_heads, layer.self_attn.d_k).transpose(1, 2)
            k = k.view(batch_size, -1, layer.self_attn.num_heads, layer.self_attn.d_k).transpose(1, 2)
            v = v.view(batch_size, -1, layer.self_attn.num_heads, layer.self_attn.d_k).transpose(1, 2)
            
            # Compute attention scores
            scores = torch.matmul(q, k.transpose(-2, -1)) / (layer.self_attn.d_k ** 0.5)
            attn_weights = layer.self_attn.softmax(scores)
            attention_weights.append(attn_weights)
            
            attn_output = torch.matmul(attn_weights, v)
            attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, layer.self_attn.d_model)
            attn_output = layer.self_attn.linear_out(attn_output)
            
            x = layer.residual1(x, lambda x: attn_output)
            x = layer.residual2(x, layer.feed_forward)
        
        x = self.model.layer_norm(x)
        output = self.model.output_linear(x)
        
        return output, attention_weights
        
    def generate(
        self,
        prompt: List[str],
        max_length: int = None,
        temperature: float = 1.0,
        top_k: int = 50,
        store_attention: bool = False
    ) -> Dict:
        """Generate text continuation from prompt"""
        self.model.eval()
        attention_weights = None
        
        # Convert prompt to tensor and print input tokens
        prompt_ids = [tiny_stories_vocab['<sos>']] + [tiny_stories_vocab[token] for token in prompt]
        # print("\nInput token IDs:", prompt_ids)
        # print("Input tokens:", ['<sos>'] + prompt)
        
        input_ids = torch.tensor(prompt_ids).unsqueeze(0).to(self.device)
        
        if max_length is None:
            max_length = self.max_seq_len - len(prompt_ids)
        else:
            max_length = min(max_length, self.max_seq_len - len(prompt_ids))
        
        generated = []
        log_probs = []
        
        with torch.no_grad():
            output = self.model(input_ids)
            
            for step in range(max_length):
                if input_ids.size(1) >= self.max_seq_len:
                    break
                
                next_token_logits = output[:, -1, :] / temperature
                
                # Print top 5 most likely next tokens before sampling
                top_logits, top_indices = torch.topk(next_token_logits[0], 5)
                # print(f"\nStep {step} - Top 5 next tokens:")
                # for logit, idx in zip(top_logits, top_indices):
                #     token = tiny_stories_vocab.get_itos()[idx]
                #     print(f"Token: {token}, Logit: {logit:.2f}")
                
                if top_k > 0:
                    indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                    next_token_logits[indices_to_remove] = float('-inf')
                
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                
                # Print selected token
                selected_token = tiny_stories_vocab.get_itos()[next_token.item()]
                # print(f"Selected token: {selected_token} (ID: {next_token.item()})")
                
                log_probs.append(torch.log(probs[0, next_token[0]]).item())
                
                if next_token.item() == tiny_stories_vocab['<eos>']:
                    break
                    
                generated.append(selected_token)
                input_ids = torch.cat([input_ids, next_token], dim=1)
                output = self.model(input_ids)
        
        perplexity = np.exp(-np.mean(log_probs)) if log_probs else float('inf')
        
        return {
            'generated_text': generated,
            'perplexity': perplexity,
            'attention_weights': attention_weights
        }

In [3]:
tokenized_full = load_from_disk("/home/anwesh/scratch/ELL8299 Project/tokenized_tiny_stories_data")
print("Tokenized dataset loaded from disk.")

Tokenized dataset loaded from disk.


In [4]:
cache_location = '/home/anwesh/scratch/hf_cache/'
dataset = load_dataset('roneneldan/TinyStories', cache_dir=cache_location)

In [5]:
train_size = len(dataset['train']) 
valid_size = len(dataset['validation'])  

In [6]:
tokenized_train = tokenized_full.select(range(0, train_size))
tokenized_valid = tokenized_full.select(range(train_size, train_size + valid_size))

In [7]:
_spacy_model = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'textcat'])

def spacy_tokenize(text):
    """Tokenize a single string using spaCy."""
    return [token.text.lower() for token in _spacy_model(text) if not token.is_space]

def batch_spacy_tokenize(batch):
    """Tokenize a batch of samples using spaCy's nlp.pipe."""
    texts = batch['text']
    return {'tokens': [[token.text.lower() for token in doc if not token.is_space] for doc in _spacy_model.pipe(texts, batch_size=512*4, n_process=1)]}

full_dataset = concatenate_datasets([dataset['train'], dataset['validation']])


In [8]:
def get_validation_prompts(validation_dataset, num_samples=50, prompt_length=5):
    """Get prompts from random validation samples"""
    # Get random indices
    random_indices = torch.randperm(len(validation_dataset))[:num_samples].tolist()
    
    # Get samples and convert to prompts
    prompts = []
    references = []
    
    for idx in random_indices:
        tokens = validation_dataset[idx]['tokens']
        # Use first prompt_length tokens as prompt
        prompt = tokens[:prompt_length]
        # Use remaining tokens as reference
        reference = tokens[prompt_length:]
        prompts.append(prompt)
        references.append(reference)
        
    return prompts, references

In [ ]:
model_path = "/home/anwesh/scratch/ELL8299 Project/model_ckpts/seq64_layers6_heads8_lr0.0003_wd0.0/best_model.pt"

generator = TextGenerator(model_path)
    
# Get random validation prompts and references
validation_prompts, reference_continuations = get_validation_prompts(
    tokenized_valid, 
    num_samples=50, 
    prompt_length=5
)

# Store results
all_results = []

num_prompts = 0

# Generate and analyze
for prompt in tqdm(validation_prompts, desc="Generating"):
    result = generator.generate(prompt, store_attention=True)
    all_results.append(result)
    print("\nPrompt:", " ".join(prompt))
    print("Generated:", " ".join(result['generated_text']))
    print(f"Perplexity: {result['perplexity']:.2f}")
    # print("BLEU score:", bleu_score([result['generated_text']], [reference_continuations[num_prompts]]))
    
    # if num_prompts % 10 == 0:
    # # Optionally visualize attention patterns
    #     if result['attention_weights'] is not None:
    #         for head_idx in range(generator.model.layers[0].self_attn.num_heads):
    #             generator.visualize_attention(prompt, result['attention_weights'], head_idx)

    # num_prompts += 1

# Calculate aggregate metrics
avg_perplexity = np.mean([r['perplexity'] for r in all_results])
print(f"\nAverage perplexity across {len(validation_prompts)} samples: {avg_perplexity:.2f}")

# Calculate BLEU score
generated_texts = [' '.join(r['generated_text']) for r in all_results]  # Join tokens into strings
reference_texts = [[' '.join(ref)] for ref in reference_continuations]  # List of lists of strings

bleu = bleu_score(generated_texts, reference_texts, max_n=4)  
print(f"BLEU score: {bleu:.4f}")

# # Generate and analyze
# for i, prompt in enumerate(tqdm(validation_prompts, desc="Generating")):
#     print(f"\n=== Sample {i+1} ===")
#     print("Input prompt:", prompt)
    
#     # Use the same parameters that worked in the test
#     result = generator.generate(
#         prompt, 
#         store_attention=True,
#         temperature=1.0,  # Use the temperature that worked in test
#         max_length=30  # Set a reasonable max length
#     )
#     all_results.append(result)
    
#     print("Prompt:", " ".join(prompt))
#     print("Generated:", " ".join(result['generated_text']))
#     print(f"Perplexity: {result['perplexity']:.2f}")
#     print("Generated length:", len(result['generated_text']))
    
#     # Early stop after a few samples to check output
#     if i == 2:  # Look at first 3 samples
#         break

# print("\nFirst few tokens from reference continuations:")
# print(reference_continuations[:3])

Generating:   2%|▏         | 1/50 [00:00<00:34,  1.42it/s]


Prompt: once upon a time ,
Generated: there was a little boy named timmy . timmy loved to ride his bike around the park . one day , timmy saw a ball and he chased it . he ran and hopped until it went over to the park . timmy felt upset because he hated the ball . the next day , timmy 's mom
Perplexity: 3.59


Generating:   4%|▍         | 2/50 [00:01<00:26,  1.78it/s]


Prompt: once upon a time ,
Generated: there was a little boy named timmy . timmy loved to play outside . one day , he went to the park to play with his friends . they played on the swings and the slide . then , timmy saw a funny animal . it was a monkey . the monkey was funny and played together .
Perplexity: 2.52


Generating:   6%|▌         | 3/50 [00:01<00:24,  1.96it/s]

...

Generating:  98%|█████████▊| 49/50 [00:22<00:00,  2.23it/s]


Prompt: once upon a time ,
Generated: there was a big , tall tree . a little bird lived in the tree and wanted to eat something tasty . the bird saw a little worm and thought about the worm that it was bigger and yummier . but the little bird was very stubborn and did n't listen to the worm . one day ,
Perplexity: 3.96


Generating: 100%|██████████| 50/50 [00:22<00:00,  2.21it/s]


Prompt: once upon a time ,
Generated: there was a little girl named lily . she loved to eat cookies and play with her toys . one day , she was feeling restless and wanted to play outside . she asked her mommy if they could go out and go to a restaurant . her mommy said they could go see a fancy restaurant with
Perplexity: 2.75

Average perplexity across 50 samples: 3.78


IndexError: index 4 is out of bounds for dimension 0 with size 4

In [ ]:
test_prompt = ["once", "upon", "a", "time", "there"]
result = generator.generate(test_prompt, store_attention=False, temperature=1.0)

# Print token IDs and their corresponding tokens
print("Generated token sequence:")
print("\nRaw generated tokens:", result['generated_text'])

# Print vocab info
print("\nVocabulary check:")
print("Vocab size:", len(tiny_stories_vocab))
print("Sample vocab tokens:", list(tiny_stories_vocab.get_itos())[:10])
print("Special tokens:")
print("<sos> index:", tiny_stories_vocab['<sos>'])
print("<eos> index:", tiny_stories_vocab['<eos>'])
print("<pad> index:", tiny_stories_vocab['<pad>'])

Generated token sequence:

Raw generated tokens: ['was', 'a', 'girl', 'named', 'lucy', '.', 'she', 'liked', 'to', 'play', 'outside', 'in', 'the', 'garden', '.', 'one', 'day', ',', 'her', 'neighbor', 'came', 'over', '.', 'lucy', 'saw', 'the', 'neighbor', 'and', 'she', 'said', 'it', 'looked', 'good', 'to', 'her', '.', '"', 'look', 'lucy', '!', 'i', "'ve", 'going', 'to', 'the', 'store', '!', '"', 'said', 'grandma', '.', 'lucy', "'s", 'mom', 'smiled', 'and', 'said', ':']

Vocabulary check:
Vocab size: 32787
Sample vocab tokens: ['<unk>', '<pad>', '<sos>', '<eos>', 'one', 'day', ',', 'a', 'little', 'girl']
Special tokens:
<sos> index: 2
<eos> index: 3
<pad> index: 1


In [ ]:
test_prompt = validation_prompts[0]  # Take first validation prompt
print("Testing prompt:", test_prompt)

result = generator.generate(
    test_prompt,
    max_length=20,
    temperature=1.0,
    store_attention=False
)

print("\nFinal generation:")
print("Prompt:", " ".join(test_prompt))
print("Generated:", " ".join(result['generated_text']))
print(f"Perplexity: {result['perplexity']:.2f}")

Testing prompt: ['once', 'upon', 'a', 'time', ',']

Final generation:
Prompt: once upon a time ,
Generated: there was a little girl named mia . mia loved to play with her toys , but one day she
Perplexity: 2.07


In [ ]:
# Debug test with direct token inspection
test_prompt = validation_prompts[0]  # Take first validation prompt
print("Testing prompt:", test_prompt)

# Get initial tokens
prompt_ids = [tiny_stories_vocab['<sos>']] + [tiny_stories_vocab[token] for token in test_prompt]
print("\nPrompt token IDs:", prompt_ids)
print("Prompt tokens:", ['<sos>'] + test_prompt)

# Check if tokens exist in vocab
print("\nChecking if prompt tokens exist in vocab:")
for token in test_prompt:
    idx = tiny_stories_vocab[token]
    print(f"Token: {token}, ID: {idx}, Reverse lookup: {tiny_stories_vocab.get_itos()[idx]}")

# Test model output directly
input_ids = torch.tensor(prompt_ids).unsqueeze(0).to(generator.device)
with torch.no_grad():
    output = generator.model(input_ids)
    next_token_logits = output[:, -1, :]
    
    # Get top 10 predictions
    top_logits, top_indices = torch.topk(next_token_logits[0], 10)
    print("\nTop 10 predictions from model:")
    for logit, idx in zip(top_logits, top_indices):
        token = tiny_stories_vocab.get_itos()[idx]
        print(f"Token: {token}, Logit: {logit:.2f}")

# Test sampling
probs = torch.softmax(next_token_logits, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)
print("\nSampled token:", tiny_stories_vocab.get_itos()[next_token.item()])

Testing prompt: ['once', 'upon', 'a', 'time', ',']

Prompt token IDs: [2, 76, 77, 7, 78, 6]
Prompt tokens: ['<sos>', 'once', 'upon', 'a', 'time', ',']

Checking if prompt tokens exist in vocab:
Token: once, ID: 76, Reverse lookup: once
Token: upon, ID: 77, Reverse lookup: upon
Token: a, ID: 7, Reverse lookup: a
Token: time, ID: 78, Reverse lookup: time
Token: ,, ID: 6, Reverse lookup: ,

Top 10 predictions from model:
Token: there, Logit: 15.89
Token: in, Logit: 12.49
Token: a, Logit: 11.94
Token: two, Logit: 9.97
Token: jack, Logit: 7.33
Token: three, Logit: 7.11
Token: little, Logit: 7.06
Token: on, Logit: 7.02
Token: an, Logit: 6.91
Token: the, Logit: 6.67

Sampled token: there
